## 09.2.1 Computational Graphs

```
date: "Block 09"
author: "Daniel Lawson"
email: dan.lawson@bristol.ac.uk
output: html_document
version: 4.0.0
```

## Portfolio 09.2.1 - Computational Graphs

* How are complex computations structured in machine learning? What are the basic operations that make optimisation automatable?
    - Reason about the role of manual structuring of computation. When is it still useful for a data scientist to worry about it?


# 0. Introduction

Here we will put to practice the ideas described in the lecture, notably:

* Computational Graphs in PyTorch, and how they are made up of
* **Map** operations, and
* **Reduce** operations,
* and how this is important.


## 0.1 Prerequisites:

In [1]:
## Install requirements, e.g.: 
# !pip install numpy, torch

In [2]:
## Might need this first:
# !pip install pipreqsnb
!pipreqsnb --savepath requirements-block09.txt block09.2.1-MapReduce.ipynb

pipreqs  --savepath requirements-block09.txt /Users/madjl/teach/dst-private/Workshops/__temp_pipreqsnb_folder
INFO: Not scanning for jupyter notebooks.
Please, verify manually the final list of requirements.txt to avoid possible dependency confusions.
INFO: Successfully saved requirements file in requirements-block09.txt


In [3]:
print("\nREQUIREMENTS:\n")
with open('requirements-block09.txt', 'r') as f:
    print(f.read())


REQUIREMENTS:

numpy==2.5.4
torch==2.9.0



In [4]:
import numpy as np
import torch

### 1.3 Accumulate example

Here we accumulate, i.e. add up all of the values in a vector, either in a vectorised or loop way.

In [5]:
seq = np.random.randint(0, 100, size=5000000)
seq.shape

(5000000,)

In [6]:
def cumsum_diff_with_accumulate(x):
     x = np.asarray(x)
     return np.add.accumulate(x)[-1]
def cumsum_diff(x):
     sum_px = x[0]
     for px in x[1:]:
         sum_px = sum_px + px
     return sum_px

In [7]:
%%time
cumsum_diff(seq)

CPU times: user 211 ms, sys: 5.28 ms, total: 216 ms
Wall time: 217 ms


np.int64(247549874)

In [8]:
%%time
cumsum_diff_with_accumulate(seq)

CPU times: user 12 ms, sys: 4.56 ms, total: 16.6 ms
Wall time: 16.1 ms


np.int64(247549874)

## 2 Mapping and Reducing

### 2.1 Map in Python

Mapping with python (and R) is straightfoward. 

Source document: http://chryswoods.com/parallel_python/map.html

In [9]:
import math

def calc_distance(point1, point2):
    """
    Function to calculate and return the distance between
    two points
    """
    
    dx2 = (point1[0] - point2[0]) ** 2
    dy2 = (point1[1] - point2[1]) ** 2
    dz2 = (point1[2] - point2[2]) ** 2
    return math.sqrt(dx2 + dy2 + dz2)

In [10]:
points1 = [(1.0,1.0,1.0), (2.0,2.0,2.0), (3.0,3.0,3.0)]
points2 = [(4.0,4.0,4.0), (5.0,5.0,5.0), (6.0,6.0,6.0)]

distances = map(calc_distance, points1, points2)

print(distances)

Python has not done the calculation! Instead it returns an object (an iterator) that would evaluate this computation. But it does so lazily.  To get the answer, we must use the result, for example, by coercing it to a list.

This behaviour is **standard** in parallel processing environments, in which the computation may be performed remotely and there may be additional remote computations to perform. By **caching** the computation, the software environment can sometimes obtain massive efficiency gains.

This is how we get the answer:

In [11]:
print(list(distances))

[5.196152422706632, 5.196152422706632, 5.196152422706632]


Another example, this time with multiple arguments.

In [12]:
def find_smallest(arg1, arg2, arg3):
    """
    Function used to return the smallest value out 
    of 'arg1', 'arg2' and 'arg3'
    """

    return min(arg1, min(arg2, arg3))

a = [1, 2, 3, 4, 5]
b = [5, 4, 3, 2, 1]
c = [1, 2, 1, 2, 1]

result = map(find_smallest, a, b, c)

In [13]:
list(result)

[1, 2, 1, 2, 1]

CHALLENGE: Generalise calc_distance so that it can accept points in any numbers of dimensions.  Generalise find_smallest so that it can accept any number of arguments.

### 2.2 Reduce in Python

This is also straightfoward, and exists anagously in R.

See http://chryswoods.com/parallel_python/reduce.html

Lets start by defining a mapping problem. We will need to define the **add** operation as a **function**, so that we can call it from map and reduce.

In [14]:
def add(x, y):
    """Function to return the sum of x and y"""
    return x + y

a = [1, 2, 3, 4, 5]
b = [6, 7, 8, 9, 10]

result = map(add, a, b)

print(list(result))

[7, 9, 11, 13, 15]


This is how we use reduce to automatically apply addition to the entire set of points.

In [15]:
from functools import reduce

result = map(add, a, b)

total = reduce(add, result)

print(total)

55


Reduce has an optional third argument which is the initial value that is used as the first value for the reduction.

In [16]:
result = map(add, a, b)
total = reduce(add, result, 10)
print(total)

65


The standard "reduce" function does *not* do anything clever with the computation tree. It simply evaluates the reduction using the sequential definition. That means that it does *not* assume commutivity, which means it can be used with other operations:

In [17]:
def join_strings(x, y):
    return "%s %s" % (x,y)
c = ["cat", "dog", "mouse", "fish"]

result = reduce(join_strings, c)
print(result)

cat dog mouse fish


### 2.2.1 accumulate vs reduce in python

Python defines **reduce** to give only the final answer, whereas **accumulate** gives the running total as a list (via an iterator, like map).

This is not a universally recognised separation!

In [18]:
from itertools import accumulate

result = map(add, a, b)
total = accumulate(result, add)
print(list(total))

[7, 16, 27, 40, 55]


In [19]:
print(list(accumulate(c, join_strings)))

['cat', 'cat dog', 'cat dog mouse', 'cat dog mouse fish']


## 3 Map-Reduce in PyTorch

Map and reduce are the two building blocks of almost every tensor operation in modern ML libraries such as PyTorch (and numpy, JAX, TensorFlow...). Knowing this helps you write code that runs much faster.

First import the libraries:

In [20]:
import torch
from functools import reduce
torch.manual_seed(1)

### 3.1 The loss function is a map-reduce

The mean squared error loss is usually written in one line:

```python
loss = torch.mean((y_pred - y_true)**2)
```

Read it from the inside out:

* `(y_pred - y_true)**2` is a **map**: the same function $f(a,b) = (a-b)^2$ is applied independently to every pair of elements. No element needs to know about any other, so this is *embarrassingly parallel*.
* `torch.mean(...)` is a **reduce**: the operation `add` is applied repeatedly to combine all the values into one number, which is then divided by $n$.

To convince ourselves, let's compute the same thing three ways: with python's `map` and `reduce` from Section 2, with numpy, and with PyTorch.

In [21]:
n = 1000
x = torch.linspace(0, 1, n)
y_true = 2 * x + 1 + 0.1 * torch.randn(n)   # data from a straight line, plus noise
y_pred = 1.5 * x + 1.2                       # some (wrong) model prediction

# 1. Explicit map and reduce, using plain python floats
def sq_err(a, b):
    return (a - b) ** 2

errs = map(sq_err, y_pred.tolist(), y_true.tolist())   # MAP
loss_mapreduce = reduce(add, errs) / n                  # REDUCE (add was defined in 2.2)

# 2. numpy
loss_numpy = np.mean((y_pred.numpy() - y_true.numpy())**2)

# 3. PyTorch
loss_torch = torch.mean((y_pred - y_true)**2)

print(loss_mapreduce, loss_numpy, loss_torch.item())

0.033743994185306395 0.033743996 0.03374399617314339


They agree (to floating point precision). The difference is *who decides how to do the work*. Python's `reduce` must go left-to-right, one element at a time. `torch.mean` only promises the answer, so it is free to split the data into chunks, sum each chunk in parallel (on CPU cores or thousands of GPU threads), and then sum the partial sums: a **tree reduction**. This is only allowed because addition is **associative**: $(a+b)+c = a+(b+c)$.

**Floating point addition is not exactly associative!** So the order of the reduction can change the answer. We can see this using the two ways of adding up from Section 1.3: `np.add.accumulate` adds strictly left to right, while `np.sum` uses pairwise (tree) summation.

In [22]:
v = np.full(10_000_000, 0.1, dtype=np.float32)   # ten million copies of 0.1; the true sum is 1,000,000

print("sequential (accumulate):", np.add.accumulate(v)[-1])
print("tree      (np.sum)     :", np.sum(v))
print("torch.sum              :", torch.sum(torch.from_numpy(v)).item())

sequential (accumulate): 1.087937e+06
tree      (np.sum)     : 1.0000001e+06
torch.sum              : 999999.625


The sequential answer is wrong: once the running total is large, adding 0.1 in 32-bit precision loses digits. The tree reduction only ever adds numbers of similar size, so the error stays small. numpy and PyTorch still disagree in the last few digits, because they split the work up differently.

Training the same neural network twice on a GPU can give (slightly) different results: the order in which thousands of threads complete their part of a reduction is not fixed.

**Some questions:** Identify the map(s) and the reduce(s) in each of the following. What is the reduce operation in each case (`add`, `max`, ...)?

1. Root mean squared error: `torch.sqrt(torch.mean((y_pred - y_true)**2))`
2. Mean absolute error: `torch.mean(torch.abs(y_pred - y_true))`
3. A dot product / linear model: `X @ w` where `X` is $n \times p$ and `w` has length $p$. (Hint: what happens for each row of `X`?)
4. `torch.max(torch.abs(y_pred - y_true))`. Could this be parallelised in the same way as the sum?

### 3.2 The computational graph: a recipe, not an answer

Python's `map` does not compute anything immediately; it returns an object that *describes* the computation and PyTorch is the same. 

For variables we have set `requires_grad=True`, every operation is **recorded** in a **computational graph**: a directed graph where each node is an operation (a map or a reduce) and the edges show which results feed into which.

Let's make the model's parameters, a slope `w` and intercept `b`, into tensors that PyTorch should track.

In [23]:
w = torch.tensor(1.5, requires_grad=True)
b = torch.tensor(1.2, requires_grad=True)

y_pred = w * x + b                      # map: one prediction per data point
loss = torch.mean((y_pred - y_true)**2) # map, then reduce

print(loss)

tensor(0.0337, grad_fn=<MeanBackward0>)


Note the `grad_fn=<MeanBackward0>` in the output. `loss` is not just a number: it remembers that it was made by a `mean`. We can follow the graph backwards from `loss` to see the whole recipe. The helper function below walks the graph recursively and prints it.

In [24]:
def print_graph(fn, depth=0):
    # Recursively print the computational graph that produced a tensor
    if fn is None:
        return
    print("    " * depth + type(fn).__name__)
    for child, _ in fn.next_functions:
        print_graph(child, depth + 1)

print_graph(loss.grad_fn)

MeanBackward0
    PowBackward0
        SubBackward0
            AddBackward0
                MulBackward0
                    AccumulateGrad
                AccumulateGrad


Read it from the top (the output) down to the leaves (the inputs):

* `MeanBackward0` is the **reduce**,
* `PowBackward0`, `SubBackward0`, `AddBackward0` and `MulBackward0` are the **maps** (each elementwise),
* `AccumulateGrad` marks a leaf we asked to track: these are `w` and `b`. (`y_true` and `x` do not appear because they do not require gradients.)

Every node is named `...Backward`, because the reason PyTorch keeps this graph is to be able to run it **backwards**.

### 3.3 The gradient is a map-reduce too

Calling `loss.backward()` runs the graph backwards and computes $\partial \text{loss} / \partial w$ and $\partial \text{loss} / \partial b$, storing them in `w.grad` and `b.grad`. 

This **automatic differentiation** (autograd) is how every neural network is trained.

In [25]:
loss.backward()
print("autograd:", w.grad.item(), b.grad.item())

autograd: -0.1337636113166809 -0.10174518823623657


Let's check by hand. With $L = \frac{1}{n}\sum_i (w x_i + b - y_i)^2$:

$$\frac{\partial L}{\partial w} = \frac{1}{n}\sum_i 2(w x_i + b - y_i)\,x_i, \qquad \frac{\partial L}{\partial b} = \frac{1}{n}\sum_i 2(w x_i + b - y_i).$$

Each of these is, again, **a map** (compute one term per data point) **followed by a reduce** (sum them up). The backward pass of a reduce is a map, and the backward pass of a map is a map: the structure is preserved.

In [26]:
## torch can be called with `no_grad()` to avoid building a computational graph.
with torch.no_grad():   # we don't need a graph for this calculation
    r = (w * x + b - y_true)            # map
    dw = torch.mean(2 * r * x)          # map then reduce
    db = torch.mean(2 * r)              # map then reduce
print("by hand :", dw.item(), db.item())

by hand : -0.1337636113166809 -0.10174518078565598


#### Why this matters: data-parallel training

Because the gradient is a **sum over data points**, we can split the data into chunks, compute the gradient of each chunk *independently* (on different cores, GPUs, or machines), and then add the results together. 

This is exactly MapReduce, and it is how large models are trained: each GPU gets a different batch (the **map**) and the gradients are then combined with an operation called **all-reduce** (the **reduce**).

Let's simulate this with 4 "workers". We use the *sum* of squared errors for each chunk, and divide by the total $n$ at the end, so that the pieces add up correctly.

In [27]:
def chunk_gradient(x_chunk, y_chunk):
    # The work done by one worker: gradient of the chunk's contribution to the loss
    w_local = torch.tensor(1.5, requires_grad=True)
    b_local = torch.tensor(1.2, requires_grad=True)
    loss_chunk = torch.sum((w_local * x_chunk + b_local - y_chunk)**2) / n
    loss_chunk.backward()
    return torch.stack([w_local.grad, b_local.grad])

chunks = zip(torch.chunk(x, 4), torch.chunk(y_true, 4))
partial_grads = map(lambda c: chunk_gradient(*c), chunks)   # MAP: one gradient per worker
total_grad = reduce(add, partial_grads)                       # REDUCE: all-reduce

print("data-parallel:", total_grad.tolist())
print("full data    :", [w.grad.item(), b.grad.item()])

data-parallel: [-0.1337636113166809, -0.10174518823623657]
full data    : [-0.1337636113166809, -0.10174518823623657]


**QUESTION:** In `chunk_gradient` we divided by the *total* `n`, not by the chunk size. What goes wrong if each worker uses `torch.mean` on its own chunk and we then add the results?

### 3.4 Putting it together: gradient descent

With the gradient available, fitting the model is a loop: build the graph (forward), run it backwards (backward), take a step downhill, repeat. 

We **reset** the gradients each step, because PyTorch *accumulates* (adds) into `.grad`. That accumulation is itself a reduce, and lets you split a batch that is too big for memory into pieces and sum the gradients.

In [28]:
w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
lr = 0.5

for step in range(200):
    loss = torch.mean((w * x + b - y_true)**2)   # forward: builds a fresh graph
    loss.backward()                              # backward: map-reduce for the gradient
    with torch.no_grad():                        # the update itself should not be recorded
        w -= lr * w.grad
        b -= lr * b.grad
    w.grad.zero_() # clear the gradient for the next step
    b.grad.zero_()
    if step % 50 == 0:
        print(step, loss.item())

print("fitted w, b:", w.item(), b.item(), "(true values 2 and 1)")

0 4.346858978271484
50 0.010704198852181435
100 0.010584529489278793
150 0.010584396310150623
fitted w, b: 1.9963510036468506 1.002697229385376 (true values 2 and 1)


### 3.5 Why should you still care, in the age of AI?

An AI assistant will happily write the training loop above for you. So why understand what is underneath? 

You are a creator of AI, not just a user. When you design new algorithms, you may want to improve:

1. **Computational efficiency.** Maps are trivially parallel. Reductions need communication between workers. A python `for` loop over the elements of a tensor hides the map from PyTorch, and runs thousands of times slower; see the timing in Section 1.3.
2. **Memory usage.** To compute gradients PyTorch must keep the intermediate result of every node. For a large model, *storing the graph* rather than the parameters may fill the GPU. This is why we use `torch.no_grad()` when we don't need gradients.
3. **Correctness can fail silently.** If any step leaves PyTorch (e.g. converting to numpy or a python float) or is `.detach()`ed, the graph is broken and no gradient flows past that point. There is often no error, just a model that doesn't learn.
4. **Reproducibility.** As we saw in 3.1, the order of a reduction changes the answer. Parallel reductions are a source of non-determinism in results.
5. **Scaling.** Training and serving large models is about splitting maps across devices and organising reductions efficiently (all-reduce, sharding, etc.).
6. Only PyTorch questions can be parallelised by PyTorch. Is it a general purpose problem solver? Why haven't you used it in other parts of the course, or for other problems?

Here is a bug of type 3. Can you spot it before running the code?

In [29]:
w = torch.tensor(1.5, requires_grad=True)
b = torch.tensor(1.2, requires_grad=True)

y_pred = w * x + b
y_pred_clipped = torch.tensor(np.clip(y_pred.detach().numpy(), 0, 3))  # "clip predictions to a sensible range"
loss = torch.mean((y_pred_clipped - y_true)**2)

print(loss)
print("loss.grad_fn:", loss.grad_fn)

tensor(0.0337)
loss.grad_fn: None


Explanation: `grad_fn` is `None`: the graph stops at the clipping step, so `loss` has no connection to `w` or `b`, so `loss.backward()` would raise an error here, and in more complex models where *some* path survives, the affected parameters would just silently receive no gradient.

**QUESTION:** Can you fix the code so gradients flow to `w` and `b`? 

* Hints: 
    * is there a PyTorch version of `np.clip`?
    * Use `print_graph` to confirm.
    * How do we populate the `w.grad` and `b.grad` terms?